# Contest Notebook: Artificial Neural Networks  
## Architecture, Forward Pass, and Backpropagation

Welcome to a **high-difficulty coding contest notebook** for Artificial Neural Networks.

This notebook is designed to test whether you truly understand:

1. **Artificial Neural Architectures**
2. **Training Neural Networks**
   - Forward pass: tensor operations
   - Backward pass: backpropagation

## Contest Rules

- This is a **from-scratch** implementation challenge.
- Use **NumPy only** for neural-network computation.
- **Do not** use PyTorch, TensorFlow, JAX, Keras, scikit-learn neural network classes, or automatic differentiation.
- You must write both:
  - **code**, and
  - **mathematical / conceptual explanations** in new Markdown cells where instructed.
- Public tests are included, but passing them does **not** guarantee full credit.
- Hidden checks used by the instructor may evaluate:
  - vectorization,
  - correctness,
  - numerical stability,
  - explanation quality,
  - code clarity.

## Deliverables inside this notebook

You must complete all required `TODO` regions and add your explanations in Markdown cells below the prompts.

## Difficulty Notice

This notebook is intentionally difficult. It is designed like a **challenge game / contest**.  
The first student to complete it **correctly** may receive additional points, subject to instructor verification.

## Allowed tools

You may use:

- `numpy`
- `matplotlib` for plots only
- Python standard library

You may **not** use:

- `torch`, `tensorflow`, `keras`, `jax`
- automatic differentiation
- high-level neural-network libraries
- copied internet solutions

## Suggested workflow

1. Complete the architecture questions first.
2. Implement the forward pass carefully.
3. Implement the backward pass and verify with gradient checking.
4. Implement the optimizer and training loop.
5. Attempt the boss-level bonus section only after the required tasks work.

---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

np.set_printoptions(precision=5, suppress=True)
GLOBAL_SEED = 7
rng = np.random.default_rng(GLOBAL_SEED)

def set_seed(seed=7):
    np.random.seed(seed)
    return np.random.default_rng(seed)

rng = set_seed(GLOBAL_SEED)
print("Environment ready.")

Environment ready.


In [2]:
# Helper datasets and utilities
def make_xor(n_per_quadrant=80, noise=0.10, seed=7):
    rng = np.random.default_rng(seed)
    centers = np.array([
        [-1.0, -1.0],
        [-1.0,  1.0],
        [ 1.0, -1.0],
        [ 1.0,  1.0]
    ])
    labels = np.array([0, 1, 1, 0])  # XOR pattern
    X_parts, y_parts = [], []
    for c, lab in zip(centers, labels):
        X_parts.append(c + noise * rng.standard_normal((n_per_quadrant, 2)))
        y_parts.append(np.full(n_per_quadrant, lab, dtype=int))
    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)
    idx = rng.permutation(len(X))
    return X[idx], y[idx]

def make_spiral(points_per_class=120, classes=3, noise=0.20, seed=7):
    rng = np.random.default_rng(seed)
    X = np.zeros((points_per_class * classes, 2))
    y = np.zeros(points_per_class * classes, dtype=int)
    for j in range(classes):
        ix = range(points_per_class * j, points_per_class * (j + 1))
        r = np.linspace(0.0, 1, points_per_class)
        t = np.linspace(j * 4, (j + 1) * 4, points_per_class) + rng.standard_normal(points_per_class) * noise
        X[ix] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[ix] = j
    idx = rng.permutation(len(X))
    return X[idx], y[idx]

def train_val_split(X, y, val_fraction=0.20, seed=7):
    rng = np.random.default_rng(seed)
    n = len(X)
    idx = rng.permutation(n)
    n_val = int(round(n * val_fraction))
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

def accuracy_from_probs(probs, y_true):
    preds = np.argmax(probs, axis=1)
    return np.mean(preds == y_true)

def plot_2d_dataset(X, y, title="dataset"):
    plt.figure(figsize=(5, 4))
    plt.scatter(X[:, 0], X[:, 1], c=y, s=18)
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

## Challenge 0 — One-hot encoding and tensor discipline

Before building a neural network, confirm that you understand shape conventions.

### Task
Implement `one_hot(y, num_classes)` such that:

- input: `y` has shape `(N,)`
- output: matrix has shape `(N, C)`
- row `i` contains a `1` at the true class and `0` elsewhere

### Constraints
- No Python loops over samples
- Must be vectorized

After the code cell, add a **Markdown explanation** answering:

1. Why is one-hot encoding useful for multi-class classification?
2. Why does shape consistency matter in the forward and backward pass?

In [ ]:
def one_hot(y, num_classes):
    # TODO: replace the next line with a vectorized implementation
    raise NotImplementedError("Implement one_hot(y, num_classes)")

In [ ]:
# Public tests for Challenge 0
y_small = np.array([2, 0, 1, 2])
Y_small = one_hot(y_small, 3)

assert isinstance(Y_small, np.ndarray), "Output must be a NumPy array."
assert Y_small.shape == (4, 3), f"Expected shape (4, 3), got {Y_small.shape}"
assert np.allclose(Y_small.sum(axis=1), 1), "Each row must sum to 1."
assert np.array_equal(np.argmax(Y_small, axis=1), y_small), "Argmax of one-hot rows should recover labels."

print("Challenge 0 public tests passed.")

### Your written response for Challenge 0

Add a **new Markdown cell below this one** and explain your answers clearly.

## Challenge 1 — Architecture Forensics

You will analyze fully-connected feedforward architectures **before** training them.

A dense layer with bias has:

$\text{params} = n_{in} \cdot n_{out} + n_{out}$

### Task A
Implement:

- `dense_param_count(n_in, n_out, use_bias=True)`
- `architecture_report(layer_dims, use_bias=True)`

where `layer_dims` is a list such as `[2, 64, 64, 3]`.

Your report must return enough information to identify:

- layer index
- input width
- output width
- parameter count per layer
- cumulative parameter count
- total parameter count

### Task B
Implement `rank_feasible_architectures(candidates, budget)` that returns the feasible architectures sorted by **descending total parameters**, breaking ties alphabetically by architecture name.

Use the candidate set below.

In [ ]:
CANDIDATES = {
    "A": [2, 64, 64, 3],
    "B": [2, 128, 32, 3],
    "C": [2, 32, 32, 32, 32, 3],
    "D": [2, 256, 3],
    "E": [2, 16, 16, 16, 16, 16, 16, 3],
}
BUDGET = 5000

def dense_param_count(n_in, n_out, use_bias=True):
    # TODO
    raise NotImplementedError("Implement dense_param_count")

def architecture_report(layer_dims, use_bias=True):
    # TODO
    # Suggested return:
    # {
    #   "layers": [
    #       {"layer": 1, "n_in": ..., "n_out": ..., "params": ..., "cumulative": ...},
    #       ...
    #   ],
    #   "total_params": ...
    # }
    raise NotImplementedError("Implement architecture_report")

def rank_feasible_architectures(candidates, budget):
    # TODO
    raise NotImplementedError("Implement rank_feasible_architectures")

In [ ]:
# Public tests for Challenge 1
assert dense_param_count(2, 3) == 9, "For bias-enabled dense layer: 2*3 + 3 = 9"
assert dense_param_count(5, 4, use_bias=False) == 20, "Without bias: 5*4 = 20"

rep = architecture_report([2, 4, 3])
assert "layers" in rep and "total_params" in rep, "Report must contain 'layers' and 'total_params'."
assert rep["total_params"] == (2*4+4) + (4*3+3), "Total params incorrect."

ranked = rank_feasible_architectures(CANDIDATES, BUDGET)
assert isinstance(ranked, list), "rank_feasible_architectures must return a list."
assert all(name in CANDIDATES for name, _ in ranked), "Returned architecture names must be valid."
assert all(total <= BUDGET for _, total in ranked), "Only feasible architectures should be returned."

for i in range(len(ranked) - 1):
    left, right = ranked[i], ranked[i + 1]
    assert left[1] >= right[1], "Architectures must be sorted by descending parameter count."

print("Challenge 1 public tests passed.")

### Your written response for Challenge 1

Add a **new Markdown cell below this one** and answer:

1. Between a shallow-wide network and a deeper-narrow network with similar parameter count, which would you expect to have stronger representational flexibility on nonlinear data, and why?
2. Why is parameter count alone not enough to predict trainability?

## Challenge 2 — Stable forward pass using tensor operations

Now implement the forward pass of a multi-layer perceptron using vectorized NumPy operations.

### Required functions

Implement:

- `init_mlp(layer_dims, seed=7, scheme="he")`
- `linear_forward(A_prev, W, b)`
- `relu(Z)`
- `tanh_act(Z)`
- `stable_softmax(Z)`
- `cross_entropy_from_probs(P, y_onehot)`
- `forward_mlp(X, params, hidden_activation="relu")`

### Shape convention

- `X`: `(N, D)`
- `W_l`: `(D_{l-1}, D_l)`
- `b_l`: `(1, D_l)`
- activations: `(N, D_l)`

### Important
Your implementation must be **numerically stable**.  
In particular, your softmax must not overflow on large logits.

In [ ]:
def init_mlp(layer_dims, seed=7, scheme="he"):
    # TODO
    # Return a dictionary like:
    # {
    #   "W1": ...,
    #   "b1": ...,
    #   "W2": ...,
    #   "b2": ...,
    #   ...
    # }
    raise NotImplementedError("Implement init_mlp")

def linear_forward(A_prev, W, b):
    # TODO
    raise NotImplementedError("Implement linear_forward")

def relu(Z):
    # TODO
    raise NotImplementedError("Implement relu")

def tanh_act(Z):
    # TODO
    raise NotImplementedError("Implement tanh_act")

def stable_softmax(Z):
    # TODO
    raise NotImplementedError("Implement stable_softmax")

def cross_entropy_from_probs(P, y_onehot, eps=1e-12):
    # TODO
    raise NotImplementedError("Implement cross_entropy_from_probs")

def forward_mlp(X, params, hidden_activation="relu"):
    # TODO
    # Return:
    # logits, probs, cache
    #
    # Suggested cache keys:
    # cache["A0"] = X
    # cache["Z1"], cache["A1"], ...
    # cache["logits"], cache["probs"]
    raise NotImplementedError("Implement forward_mlp")

In [ ]:
# Public tests for Challenge 2
set_seed(7)

params = init_mlp([2, 5, 3], seed=7, scheme="he")
X_toy = np.array([[1.0, -1.0], [0.5, 2.0], [-1.5, 0.3]])

Z = linear_forward(X_toy, params["W1"], params["b1"])
assert Z.shape == (3, 5), f"Expected shape (3, 5), got {Z.shape}"

R = relu(np.array([[-1.0, 2.0, 0.0]]))
assert np.array_equal(R, np.array([[0.0, 2.0, 0.0]])), "ReLU implementation incorrect."

T = tanh_act(np.array([[0.0, 1.0]]))
assert np.allclose(T, np.tanh(np.array([[0.0, 1.0]]))), "tanh implementation incorrect."

big_logits = np.array([[1000.0, 1001.0, 1002.0]])
P_big = stable_softmax(big_logits)
assert np.all(np.isfinite(P_big)), "Softmax must be numerically stable."
assert np.allclose(P_big.sum(axis=1), 1.0), "Softmax rows must sum to 1."

y_toy = np.array([2, 1, 0])
Y_toy = one_hot(y_toy, 3)
logits, probs, cache = forward_mlp(X_toy, params, hidden_activation="relu")
loss = cross_entropy_from_probs(probs, Y_toy)

assert logits.shape == (3, 3), "Incorrect logits shape."
assert probs.shape == (3, 3), "Incorrect probabilities shape."
assert np.isfinite(loss), "Loss must be finite."

print("Challenge 2 public tests passed.")

### Your written response for Challenge 2

Add a **new Markdown cell below this one** and explain:

1. Why do we subtract the row-wise maximum before softmax?
2. Why is matrix multiplication preferred over looping through samples one at a time?
3. What is the role of initialization in the forward pass?

## Challenge 3 — Manual backward pass with gradient checking

You will now implement backpropagation **by hand**.

### Required functions

Implement:

- `relu_backward(dA, Z)`
- `tanh_backward(dA, Z)`
- `linear_backward(dZ, A_prev, W)`
- `backward_mlp(params, cache, y_onehot, hidden_activation="relu", l2_lambda=0.0)`

### Expected idea

For softmax + cross-entropy:

$\frac{\partial \mathcal{L}}{\partial Z^{(L)}} = \frac{P - Y}{N}$

Then propagate backward layer by layer.

### Important
Your gradients must have the correct shapes and must pass numerical gradient checking.

In [ ]:
def relu_backward(dA, Z):
    # TODO
    raise NotImplementedError("Implement relu_backward")

def tanh_backward(dA, Z):
    # TODO
    raise NotImplementedError("Implement tanh_backward")

def linear_backward(dZ, A_prev, W):
    # TODO
    # Return dA_prev, dW, db
    raise NotImplementedError("Implement linear_backward")

def backward_mlp(params, cache, y_onehot, hidden_activation="relu", l2_lambda=0.0):
    # TODO
    # Return grads dictionary with keys dW1, db1, dW2, db2, ...
    raise NotImplementedError("Implement backward_mlp")

In [ ]:
# Helpers for gradient checking
def l2_penalty(params):
    total = 0.0
    L = len([k for k in params if k.startswith("W")])
    for l in range(1, L + 1):
        total += np.sum(params[f"W{l}"] ** 2)
    return total

def total_loss_and_cache(X, y_onehot, params, hidden_activation="relu", l2_lambda=0.0):
    logits, probs, cache = forward_mlp(X, params, hidden_activation=hidden_activation)
    data_loss = cross_entropy_from_probs(probs, y_onehot)
    reg_loss = 0.5 * l2_lambda * l2_penalty(params)
    return data_loss + reg_loss, cache, probs

def numerical_gradient(param_name, params, X, y_onehot, hidden_activation="relu", l2_lambda=0.0, eps=1e-5):
    grad = np.zeros_like(params[param_name])
    it = np.nditer(params[param_name], flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        idx = it.multi_index
        original = params[param_name][idx]

        params[param_name][idx] = original + eps
        loss_plus, _, _ = total_loss_and_cache(X, y_onehot, params, hidden_activation, l2_lambda)

        params[param_name][idx] = original - eps
        loss_minus, _, _ = total_loss_and_cache(X, y_onehot, params, hidden_activation, l2_lambda)

        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
        params[param_name][idx] = original
        it.iternext()
    return grad

def relative_error(a, b, eps=1e-12):
    return np.max(np.abs(a - b) / np.maximum(eps, np.abs(a) + np.abs(b)))

In [ ]:
# Public tests for Challenge 3
set_seed(11)
X_gc = np.array([
    [ 0.10, -0.20],
    [ 0.00,  0.30],
    [ 0.40, -0.50],
    [-0.30,  0.20],
])
y_gc = np.array([0, 1, 1, 0])
Y_gc = one_hot(y_gc, 2)

params_gc = init_mlp([2, 4, 2], seed=11, scheme="xavier")
loss_gc, cache_gc, probs_gc = total_loss_and_cache(X_gc, Y_gc, params_gc, hidden_activation="tanh", l2_lambda=1e-3)
grads_gc = backward_mlp(params_gc, cache_gc, Y_gc, hidden_activation="tanh", l2_lambda=1e-3)

required_keys = {"dW1", "db1", "dW2", "db2"}
assert required_keys.issubset(grads_gc.keys()), f"Missing gradient keys. Expected at least {required_keys}"

for key in ["W1", "b1", "W2", "b2"]:
    num_g = numerical_gradient(key, params_gc, X_gc, Y_gc, hidden_activation="tanh", l2_lambda=1e-3)
    ana_g = grads_gc["d" + key]
    err = relative_error(num_g, ana_g)
    print(f"{key}: relative error = {err:.3e}")
    assert err < 1e-5, f"Gradient check failed for {key}; relative error too high."

print("Challenge 3 public tests passed.")

### Your written response for Challenge 3

Add a **new Markdown cell below this one** and explain:

1. Why does the derivative of ReLU depend on the sign of the pre-activation?
2. Why is gradient checking useful?
3. Why should gradient checking not be used as the training algorithm itself?

## Challenge 4 — Optimization and training loop

You now have the components needed to train a neural network.

### Required functions

Implement:

- `iterate_minibatches(X, y, batch_size, shuffle=True, seed=7)`
- `sgd_momentum_step(params, grads, velocity, lr, momentum=0.9)`
- `train_classifier(...)`

### Training target
Train a classifier on the XOR dataset.

A correct solution should achieve very high validation accuracy on XOR because the pattern is nonlinear but low-dimensional.

### Minimum expected evidence
Your training code should print or store:

- epoch number
- train loss
- validation loss
- train accuracy
- validation accuracy

In [ ]:
def iterate_minibatches(X, y, batch_size, shuffle=True, seed=7):
    # TODO
    # Yield X_batch, y_batch
    raise NotImplementedError("Implement iterate_minibatches")

def sgd_momentum_step(params, grads, velocity, lr, momentum=0.9):
    # TODO
    # Update params in place or return updated params/velocity
    raise NotImplementedError("Implement sgd_momentum_step")

def train_classifier(
    X_train,
    y_train,
    X_val,
    y_val,
    layer_dims,
    hidden_activation="tanh",
    init_scheme="xavier",
    lr=0.1,
    momentum=0.9,
    batch_size=32,
    epochs=300,
    l2_lambda=0.0,
    seed=7,
    verbose=True
):
    # TODO
    # Suggested return:
    # params, history
    raise NotImplementedError("Implement train_classifier")

In [ ]:
# Public run for Challenge 4
set_seed(21)
X_xor, y_xor = make_xor(n_per_quadrant=90, noise=0.12, seed=21)
Xtr, Xva, ytr, yva = train_val_split(X_xor, y_xor, val_fraction=0.20, seed=21)

plot_2d_dataset(Xtr, ytr, title="XOR training split")

params_xor, history_xor = train_classifier(
    Xtr, ytr, Xva, yva,
    layer_dims=[2, 16, 16, 2],
    hidden_activation="tanh",
    init_scheme="xavier",
    lr=0.15,
    momentum=0.9,
    batch_size=32,
    epochs=300,
    l2_lambda=1e-4,
    seed=21,
    verbose=False
)

_, p_train, _ = forward_mlp(Xtr, params_xor, hidden_activation="tanh")
_, p_val, _ = forward_mlp(Xva, params_xor, hidden_activation="tanh")

train_acc = accuracy_from_probs(p_train, ytr)
val_acc = accuracy_from_probs(p_val, yva)

print(f"train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

assert train_acc >= 0.97, "Training accuracy on XOR is too low."
assert val_acc >= 0.95, "Validation accuracy on XOR is too low."

assert isinstance(history_xor, dict), "History must be a dictionary."
for key in ["train_loss", "val_loss", "train_acc", "val_acc"]:
    assert key in history_xor, f"Missing history key: {key}"

print("Challenge 4 public tests passed.")

In [ ]:
# Plot training curves after Challenge 4
plt.figure(figsize=(6, 4))
plt.plot(history_xor["train_loss"], label="train_loss")
plt.plot(history_xor["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("XOR loss curve")
plt.legend()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(history_xor["train_acc"], label="train_acc")
plt.plot(history_xor["val_acc"], label="val_acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("XOR accuracy curve")
plt.legend()
plt.show()

### Your written response for Challenge 4

Add a **new Markdown cell below this one** and explain:

1. Why can a purely linear classifier fail on XOR?
2. What does momentum do during optimization?
3. How can you tell from the curves whether the model is underfitting, overfitting, or learning appropriately?

## Challenge 5 — Boss Level Bonus: residual architecture on spiral data

This section is intentionally hard.

You will implement a small **residual block** and compare it with a plain deep network on a spiral dataset.

### Residual block definition

For an input activation $A$ with width $d$:

1. $Z_1 = AW_1 + b_1$
2. $H = \phi(Z_1)$
3. $Z_2 = HW_2 + b_2$
4. $R = A + Z_2$
5. $A_{out} = \phi(R)$

Assume all dimensions are compatible.

### Bonus tasks

Implement:

- `init_residual_block(width, seed=7, scheme="he")`
- `residual_block_forward(A, block_params, activation="relu")`
- `residual_block_backward(dA_out, cache, block_params, activation="relu")`

Then construct a classifier:

- input projection: `2 -> 64`
- two residual blocks of width `64`
- output layer: `64 -> 3`

Train on spiral data and compare to a plain MLP with similar width.

### What to submit for the bonus
Add Markdown explanations discussing:

1. Why skip connections may help gradient flow
2. Whether your residual network trained more easily than the plain network
3. What you observed about optimization difficulty

In [ ]:
def init_residual_block(width, seed=7, scheme="he"):
    # TODO (Bonus)
    raise NotImplementedError("Bonus: implement init_residual_block")

def residual_block_forward(A, block_params, activation="relu"):
    # TODO (Bonus)
    # Return A_out, cache
    raise NotImplementedError("Bonus: implement residual_block_forward")

def residual_block_backward(dA_out, cache, block_params, activation="relu"):
    # TODO (Bonus)
    # Return dA_in, grads
    raise NotImplementedError("Bonus: implement residual_block_backward")

In [ ]:
# Optional playground for the bonus section
X_sp, y_sp = make_spiral(points_per_class=120, classes=3, noise=0.20, seed=7)
plot_2d_dataset(X_sp, y_sp, title="Spiral dataset for bonus section")
print("Bonus section has no public autograder asserts. Instructor evaluation applies.")

## Final reflection

Add one final Markdown cell at the end of your notebook and answer:

1. Which part of neural-network implementation was most error-prone for you: architecture design, forward pass, or backward pass?
2. What is one debugging strategy that helped you?
3. What is one conceptual insight you gained about backpropagation?

---
**End of contest notebook.**